# ====================================================
# 1. HAM CSI - USER CLASSIFICATION
# ====================================================

In [39]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y_user = np.load("../src/y_user_new_identity.npy")
y_gesture = np.load("../src/y_gesture_new_identity.npy")

print(X.shape)
print(y_user.shape)
print(y_gesture.shape)

(1500, 200, 64)
(1500,)
(1500,)


In [40]:
still_idx = np.where(
    (y_gesture == 0) |
    (y_gesture == 4)
)[0]

X_still = X[still_idx]
y_still = y_user[still_idx]

print(X_still.shape)
print(np.unique(y_still, return_counts=True))

(600, 200, 64)
(array([0, 1, 2, 3, 4], dtype=int32), array([ 75,  75,  75,  75, 300]))


In [41]:
np.random.seed(42)

balanced_idx = []

for cls in np.unique(y_still):

    cls_idx = np.where(y_still == cls)[0]

    if cls == 4:
        cls_idx = np.random.choice(
            cls_idx,
            size=75,
            replace=False
        )

    balanced_idx.extend(cls_idx)

balanced_idx = np.array(balanced_idx)

X_still = X_still[balanced_idx]
y_still = y_still[balanced_idx]

print(X_still.shape)
print(np.unique(y_still, return_counts=True))

(375, 200, 64)
(array([0, 1, 2, 3, 4], dtype=int32), array([75, 75, 75, 75, 75]))


In [42]:
X_still = np.clip(
    X_still,
    -150,
    150
)

print(X_still.shape)

(375, 200, 64)


In [43]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y_still, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_still,
    y_cat,
    test_size=0.2,
    stratify=y_still,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(300, 200, 64)
(75, 200, 64)


In [44]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    BatchNormalization,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(
    Conv1D(
        64,
        kernel_size=5,
        activation="relu",
        input_shape=(200,64)
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        128,
        kernel_size=3,
        activation="relu"
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(GlobalAveragePooling1D())

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.4))

model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(5, activation="softmax"))

model.compile(
    optimizer=Adam(0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_8 (Conv1D)               │ (None, 196, 64)        │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 196, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_8 (MaxPooling1D)  │ (None, 98, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (None, 96, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 96, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_9 (MaxPooling1D)  │ (None, 48, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_4      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71,109 (277.77 KB)

 Trainable params: 70,725 (276.27 KB)

 Non-trainable params: 384 (1.50 KB)

In [45]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    callbacks=[early_stop]
)

Epoch 1/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - accuracy: 0.2667 - loss: 1.6727 - val_accuracy: 0.1833 - val_loss: 8.6672
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3792 - loss: 1.5337 - val_accuracy: 0.1833 - val_loss: 6.0526
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.3458 - loss: 1.4733 - val_accuracy: 0.1833 - val_loss: 6.4980
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3833 - loss: 1.3678 - val_accuracy: 0.2333 - val_loss: 3.1074
Epoch 5/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.4750 - loss: 1.2395 - val_accuracy: 0.2333 - val_loss: 2.5621
Epoch 6/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.4792 - loss: 1.2329 - val_accuracy: 0.3667 - val_loss: 2.5565
Epoch 7/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5000 - loss: 1.1955 - val_accuracy: 0.3000 - val_loss: 2.3446
Epoch 8/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.5250 - loss: 1.1042 - val_accuracy: 0.

In [46]:
loss, acc = model.evaluate(
    X_test,
    y_test
)

print("Accuracy:", acc)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.4133 - loss: 1.4954
Accuracy: 0.41333332657814026


In [47]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

y_pred = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

print(confusion_matrix(y_true, y_pred))

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "S01",
            "S02",
            "S03",
            "S04",
            "empty"
        ]
    )
)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
[[ 1  6  5  3  0]
 [ 0 13  0  0  2]
 [ 0  2  7  3  3]
 [ 0  7  2  1  5]
 [ 0  2  1  3  9]]
              precision    recall  f1-score   support

         S01       1.00      0.07      0.12        15
         S02       0.43      0.87      0.58        15
         S03       0.47      0.47      0.47        15
         S04       0.10      0.07      0.08        15
       empty       0.47      0.60      0.53        15

    accuracy                           0.41        75
   macro avg       0.49      0.41      0.36        75
weighted avg       0.49      0.41      0.36        75



# ====================================================
# 2. DELTA CSI - USER CLASSIFICATION
# ====================================================

In [48]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y_user = np.load("../src/y_user_new_identity.npy")

print("X:", X.shape)
print("y_user:", y_user.shape)

print(np.unique(y_user, return_counts=True))

X: (1500, 200, 64)
y_user: (1500,)
(array([0, 1, 2, 3, 4], dtype=int32), array([300, 300, 300, 300, 300]))


In [49]:
X_delta = np.diff(X, axis=1)

X_delta = np.clip(
    X_delta,
    -50,
    50
)

print("X_delta:", X_delta.shape)

print("Min:", X_delta.min())
print("Max:", X_delta.max())
print("Mean:", X_delta.mean())
print("Std:", X_delta.std())

X_delta: (1500, 199, 64)
Min: -50.0
Max: 50.0
Mean: 0.00033342352
Std: 1.9748566


In [50]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y_user, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_delta,
    y_cat,
    test_size=0.2,
    stratify=y_user,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1200, 199, 64)
(300, 199, 64)


In [51]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    BatchNormalization,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(
    Conv1D(
        64,
        kernel_size=5,
        activation="relu",
        input_shape=(199, 64)
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        128,
        kernel_size=3,
        activation="relu"
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(GlobalAveragePooling1D())

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.4))

model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(5, activation="softmax"))

model.compile(
    optimizer=Adam(0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_10 (Conv1D)              │ (None, 195, 64)        │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 195, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_10 (MaxPooling1D) │ (None, 97, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ (None, 95, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 95, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_11 (MaxPooling1D) │ (None, 47, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_5      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71,109 (277.77 KB)

 Trainable params: 70,725 (276.27 KB)

 Non-trainable params: 384 (1.50 KB)

In [52]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - accuracy: 0.2260 - loss: 1.7343 - val_accuracy: 0.2083 - val_loss: 1.7281
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.3302 - loss: 1.5668 - val_accuracy: 0.2333 - val_loss: 2.0174
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.3771 - loss: 1.4963 - val_accuracy: 0.2542 - val_loss: 1.8038
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.4094 - loss: 1.3988 - val_accuracy: 0.2625 - val_loss: 1.7906
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.4219 - loss: 1.4473 - val_accuracy: 0.4083 - val_loss: 1.4327
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.4365 - loss: 1.3338 - val_accuracy: 0.4000 - val_loss: 1.4489
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.4354 - loss: 1.4383 - val_accuracy: 0.3375 - val_loss: 1.6732
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.4531 - loss: 1.4359 - val_accuracy: 0.

In [53]:
loss, acc = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("Delta CSI Test Accuracy:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5033 - loss: 1.1376
Delta CSI Test Accuracy: 0.503333330154419


In [54]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

y_pred_prob = model.predict(X_test)

y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

print(confusion_matrix(y_true, y_pred))

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "S01",
            "S02",
            "S03",
            "S04",
            "empty"
        ]
    )
)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
[[39  8  3 10  0]
 [ 6  8 32  9  5]
 [ 6  4 33 16  1]
 [ 3  7 12 34  4]
 [ 0  6  8  9 37]]
              precision    recall  f1-score   support

         S01       0.72      0.65      0.68        60
         S02       0.24      0.13      0.17        60
         S03       0.38      0.55      0.45        60
         S04       0.44      0.57      0.49        60
       empty       0.79      0.62      0.69        60

    accuracy                           0.50       300
   macro avg       0.51      0.50      0.50       300
weighted avg       0.51      0.50      0.50       300



# ====================================================
# 3. DELTA CSI + RANDOM FOREST
# ====================================================

In [55]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y = np.load("../src/y_user_new_identity.npy")

X_delta = np.diff(X, axis=1)
X_delta = np.clip(X_delta, -50, 50)

print(X_delta.shape)

(1500, 199, 64)


In [56]:
features = []

for sample in X_delta:

    mean_feat = np.mean(sample, axis=0)
    std_feat = np.std(sample, axis=0)

    min_feat = np.min(sample, axis=0)
    max_feat = np.max(sample, axis=0)

    energy_feat = np.sum(sample**2, axis=0)

    feature_vector = np.concatenate([
        mean_feat,
        std_feat,
        min_feat,
        max_feat,
        energy_feat
    ])

    features.append(feature_vector)

X_feat = np.array(features)

print(X_feat.shape)

(1500, 320)


In [57]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_feat,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(1200, 320)
(300, 320)


In [58]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

,n_estimators,500
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [59]:
acc = rf.score(X_test, y_test)

print("RF Accuracy:", acc)

RF Accuracy: 0.4766666666666667


# ====================================================
# 4. Delta CSI + Normalizasyon
# ====================================================

In [60]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y = np.load("../src/y_user_new_identity.npy")

print(X.shape)
print(y.shape)

(1500, 200, 64)
(1500,)


In [61]:
X_delta = np.diff(X, axis=1)

mean = X_delta.mean(axis=(1,2), keepdims=True)
std = X_delta.std(axis=(1,2), keepdims=True)

X_delta = (X_delta - mean) / (std + 1e-8)

X_delta = np.clip(X_delta, -5, 5)

print(X_delta.shape)

print("Min:", X_delta.min())
print("Max:", X_delta.max())
print("Mean:", X_delta.mean())
print("Std:", X_delta.std())

(1500, 199, 64)
Min: -5.0
Max: 5.0
Mean: -1.2752776e-06
Std: 0.96295875


In [62]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_delta,
    y_cat,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1200, 199, 64)
(300, 199, 64)


In [63]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    BatchNormalization,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(
    Conv1D(
        64,
        kernel_size=5,
        activation="relu",
        input_shape=(199,64)
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        128,
        kernel_size=3,
        activation="relu"
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(GlobalAveragePooling1D())

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.4))

model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(5, activation="softmax"))

model.compile(
    optimizer=Adam(0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_12 (Conv1D)              │ (None, 195, 64)        │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 195, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_12 (MaxPooling1D) │ (None, 97, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_13 (Conv1D)              │ (None, 95, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 95, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_13 (MaxPooling1D) │ (None, 47, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_6      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71,109 (277.77 KB)

 Trainable params: 70,725 (276.27 KB)

 Non-trainable params: 384 (1.50 KB)

In [64]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 0.2510 - loss: 1.7182 - val_accuracy: 0.2458 - val_loss: 1.6219
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.3365 - loss: 1.5321 - val_accuracy: 0.2042 - val_loss: 1.6625
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.3885 - loss: 1.4466 - val_accuracy: 0.2083 - val_loss: 1.7088
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.4240 - loss: 1.3750 - val_accuracy: 0.3583 - val_loss: 1.4337
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.4333 - loss: 1.3440 - val_accuracy: 0.3583 - val_loss: 1.4980
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.4729 - loss: 1.2993 - val_accuracy: 0.3958 - val_loss: 1.4025
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.4833 - loss: 1.2726 - val_accuracy: 0.3750 - val_loss: 1.3940
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.4719 - loss: 1.2825 - val_accuracy: 0.

In [65]:
loss, acc = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("Normalized Delta Accuracy:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5067 - loss: 1.4043
Normalized Delta Accuracy: 0.5066666603088379


In [66]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

y_pred_prob = model.predict(X_test)

y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

print(confusion_matrix(y_true, y_pred))

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "S01",
            "S02",
            "S03",
            "S04",
            "empty"
        ]
    )
)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
[[40 10  0  7  3]
 [ 5 22 14  8 11]
 [11 19 15  8  7]
 [ 7  9  3 25 16]
 [ 0  6  0  4 50]]
              precision    recall  f1-score   support

         S01       0.63      0.67      0.65        60
         S02       0.33      0.37      0.35        60
         S03       0.47      0.25      0.33        60
         S04       0.48      0.42      0.45        60
       empty       0.57      0.83      0.68        60

    accuracy                           0.51       300
   macro avg       0.50      0.51      0.49       300
weighted avg       0.50      0.51      0.49       300



 # ====================================================
# 4.  Delta CSI + CNN-LSTM
# ====================================================

In [73]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y = np.load("../src/y_user_new_identity.npy")

X_delta = np.diff(X, axis=1)
X_delta = np.clip(X_delta, -50, 50)

print(X_delta.shape)

(1500, 199, 64)


In [74]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_delta,
    y_cat,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1200, 199, 64)
(300, 199, 64)


In [75]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    BatchNormalization,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

model = Sequential()

# CNN kısmı
model.add(
    Conv1D(
        filters=64,
        kernel_size=5,
        activation="relu",
        input_shape=(199,64)
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        filters=128,
        kernel_size=3,
        activation="relu"
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

# LSTM kısmı
model.add(
    LSTM(
        64,
        return_sequences=False
    )
)

model.add(Dropout(0.4))

# Dense
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(5, activation="softmax"))

model.compile(
    optimizer=Adam(0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_16 (Conv1D)              │ (None, 195, 64)        │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 195, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_16 (MaxPooling1D) │ (None, 97, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_17 (Conv1D)              │ (None, 95, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 95, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_17 (MaxPooling1D) │ (None, 47, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 99,909 (390.27 KB)

 Trainable params: 99,525 (388.77 KB)

 Non-trainable params: 384 (1.50 KB)

In [76]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 50ms/step - accuracy: 0.2375 - loss: 1.6508 - val_accuracy: 0.2208 - val_loss: 1.6037
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.3250 - loss: 1.5477 - val_accuracy: 0.3125 - val_loss: 1.5983
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.3406 - loss: 1.5014 - val_accuracy: 0.2625 - val_loss: 1.5696
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.4031 - loss: 1.3926 - val_accuracy: 0.3083 - val_loss: 1.5232
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.4323 - loss: 1.3560 - val_accuracy: 0.3125 - val_loss: 1.4728
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.4677 - loss: 1.2714 - val_accuracy: 0.3458 - val_loss: 1.5226
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.5115 - loss: 1.1742 - val_accuracy: 0.3167 - val_loss: 1.5581
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.5448 - loss: 1.1140 - val_accuracy: 0.

KeyboardInterrupt: 

In [ ]:
loss, acc = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("Delta CNN-LSTM Accuracy:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4100 - loss: 2.1395
Delta CNN-LSTM Accuracy: 0.4099999964237213


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_pred_prob = model.predict(X_test)

y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

print(confusion_matrix(y_true, y_pred))

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "S01",
            "S02",
            "S03",
            "S04",
            "empty"
        ]
    )
)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
[[30  9  9 10  2]
 [ 7 14 15 13 11]
 [ 5 18 17 14  6]
 [10  9 10 21 10]
 [ 3  2  4 10 41]]
              precision    recall  f1-score   support

         S01       0.55      0.50      0.52        60
         S02       0.27      0.23      0.25        60
         S03       0.31      0.28      0.30        60
         S04       0.31      0.35      0.33        60
       empty       0.59      0.68      0.63        60

    accuracy                           0.41       300
   macro avg       0.40      0.41      0.41       300
weighted avg       0.40      0.41      0.41       300



 # ====================================================
# 4.  Delta CSI + Gesture Classification
# ====================================================

In [77]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y_gesture = np.load("../src/y_gesture_new_identity.npy")

X_delta = np.diff(X, axis=1)
X_delta = np.clip(X_delta, -50, 50)

print("X_delta:", X_delta.shape)
print("Gesture dağılımı:", np.unique(y_gesture, return_counts=True))

X_delta: (1500, 199, 64)
Gesture dağılımı: (array([0, 1, 2, 3, 4], dtype=int32), array([300, 300, 300, 300, 300]))


In [78]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y_gesture, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_delta,
    y_cat,
    test_size=0.2,
    stratify=y_gesture,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1200, 199, 64)
(300, 199, 64)


In [79]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, BatchNormalization
from tensorflow.keras.layers import GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(Conv1D(64, kernel_size=5, activation="relu", input_shape=(199,64)))
model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(Conv1D(128, kernel_size=3, activation="relu"))
model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(GlobalAveragePooling1D())

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.4))

model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(5, activation="softmax"))

model.compile(
    optimizer=Adam(0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_18 (Conv1D)              │ (None, 195, 64)        │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 195, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_18 (MaxPooling1D) │ (None, 97, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_19 (Conv1D)              │ (None, 95, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 95, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_19 (MaxPooling1D) │ (None, 47, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_7      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71,109 (277.77 KB)

 Trainable params: 70,725 (276.27 KB)

 Non-trainable params: 384 (1.50 KB)

In [80]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.2115 - loss: 1.8204 - val_accuracy: 0.2000 - val_loss: 2.1037
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.2698 - loss: 1.6814 - val_accuracy: 0.2000 - val_loss: 1.6588
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.3198 - loss: 1.6155 - val_accuracy: 0.2000 - val_loss: 1.8110
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.2927 - loss: 1.5951 - val_accuracy: 0.2167 - val_loss: 1.6725
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.3354 - loss: 1.5598 - val_accuracy: 0.2750 - val_loss: 1.5905
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.3229 - loss: 1.5547 - val_accuracy: 0.2542 - val_loss: 1.6919
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.3323 - loss: 1.5959 - val_accuracy: 0.3333 - val_loss: 1.5342
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.3604 - loss: 1.5487 - val_accuracy: 0.

In [81]:
loss, acc = model.evaluate(X_test, y_test, verbose=1)

print("Delta Gesture Accuracy:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3133 - loss: 1.9721
Delta Gesture Accuracy: 0.31333333253860474


In [82]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

gesture_names = ["still", "hand_clap", "horizontal_arm_wave", "bend", "empty"]

y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=gesture_names))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
[[14 23 20  0  3]
 [24 24 10  0  2]
 [12 23 22  0  3]
 [15 31 12  0  2]
 [ 4  9 13  0 34]]
                     precision    recall  f1-score   support

              still       0.20      0.23      0.22        60
          hand_clap       0.22      0.40      0.28        60
horizontal_arm_wave       0.29      0.37      0.32        60
               bend       0.00      0.00      0.00        60
              empty       0.77      0.57      0.65        60

           accuracy                           0.31       300
          macro avg       0.30      0.31      0.29       300
       weighted avg       0.30      0.31      0.29       300



/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

#################

In [83]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y = np.load("../src/y_user_new_identity.npy")

X_delta = np.diff(X, axis=1)
X_delta = np.clip(X_delta, -50, 50)

print(X_delta.shape)

(1500, 199, 64)


In [84]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_delta,
    y_cat,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [85]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(
    Conv1D(
        64,
        kernel_size=7,
        activation="relu",
        input_shape=(199,64)
    )
)
model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        128,
        kernel_size=5,
        activation="relu"
    )
)
model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        256,
        kernel_size=3,
        activation="relu"
    )
)
model.add(BatchNormalization())

model.add(GlobalAveragePooling1D())

model.add(Dense(256, activation="relu"))
model.add(Dropout(0.5))

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.4))

model.add(Dense(5, activation="softmax"))

model.compile(
    optimizer=Adam(0.0005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_20 (Conv1D)              │ (None, 193, 64)        │        28,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 193, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_20 (MaxPooling1D) │ (None, 96, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_21 (Conv1D)              │ (None, 92, 128)        │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_21          │ (None, 92, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_21 (MaxPooling1D) │ (None, 46, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_22 (Conv1D)              │ (None, 44, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_22          │ (None, 44, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_8      │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 269,509 (1.03 MB)

 Trainable params: 268,613 (1.02 MB)

 Non-trainable params: 896 (3.50 KB)

In [86]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=12,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - accuracy: 0.2573 - loss: 1.6572 - val_accuracy: 0.1792 - val_loss: 1.7591
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.3479 - loss: 1.4984 - val_accuracy: 0.2542 - val_loss: 1.6233
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.4042 - loss: 1.4023 - val_accuracy: 0.2208 - val_loss: 1.6122
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.4323 - loss: 1.3911 - val_accuracy: 0.2542 - val_loss: 1.5276
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.4604 - loss: 1.3736 - val_accuracy: 0.3083 - val_loss: 1.5745
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.4625 - loss: 1.4891 - val_accuracy: 0.4208 - val_loss: 1.4533
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.4500 - loss: 1.5876 - val_accuracy: 0.4042 - val_loss: 1.5351
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.4896 - loss: 1.6574 - val_accuracy: 0.

In [87]:
loss, acc = model.evaluate(X_test, y_test)

print("Improved Delta CNN Accuracy:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5567 - loss: 3.6381
Improved Delta CNN Accuracy: 0.5566666722297668


In [88]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_pred_prob = model.predict(X_test)

y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

class_names = ["S01", "S02", "S03", "S04", "empty"]

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
[[45  6  2  6  1]
 [ 9 16 15  9 11]
 [ 8  8 28 11  5]
 [10  1  7 30 12]
 [ 1  2  2  7 48]]
              precision    recall  f1-score   support

         S01       0.62      0.75      0.68        60
         S02       0.48      0.27      0.34        60
         S03       0.52      0.47      0.49        60
         S04       0.48      0.50      0.49        60
       empty       0.62      0.80      0.70        60

    accuracy                           0.56       300
   macro avg       0.54      0.56      0.54       300
weighted avg       0.54      0.56      0.54       300



In [89]:
model.save("../models/IMPROVED_DELTA_CNN_USER_56.h5")

 # ====================================================
# Delta CSI + Improved CNN + Augmentation
# ====================================================

In [90]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y = np.load("../src/y_user_new_identity.npy")

X_delta = np.diff(X, axis=1)
X_delta = np.clip(X_delta, -50, 50)

print(X_delta.shape)

(1500, 199, 64)


In [91]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_delta,
    y_cat,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)

(1200, 199, 64)


In [92]:
noise = np.random.normal(
    0,
    1.0,
    X_train.shape
)

X_noise = X_train + noise

print(X_noise.shape)

(1200, 199, 64)


In [93]:
X_shift = np.copy(X_train)

for i in range(len(X_shift)):
    shift = np.random.randint(-5, 6)
    X_shift[i] = np.roll(
        X_shift[i],
        shift,
        axis=0
    )

print(X_shift.shape)

(1200, 199, 64)


In [94]:
X_train_aug = np.concatenate([
    X_train,
    X_noise,
    X_shift
])

y_train_aug = np.concatenate([
    y_train,
    y_train,
    y_train
])

print(X_train_aug.shape)
print(y_train_aug.shape)

(3600, 199, 64)
(3600, 5)


In [95]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(
    Conv1D(
        64,
        kernel_size=7,
        activation="relu",
        input_shape=(199,64)
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        128,
        kernel_size=5,
        activation="relu"
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        256,
        kernel_size=3,
        activation="relu"
    )
)

model.add(BatchNormalization())

model.add(GlobalAveragePooling1D())

model.add(Dense(256, activation="relu"))
model.add(Dropout(0.5))

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.4))

model.add(Dense(5, activation="softmax"))

model.compile(
    optimizer=Adam(0.0005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [96]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=12,
    restore_best_weights=True
)

history = model.fit(
    X_train_aug,
    y_train_aug,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 7s 43ms/step - accuracy: 0.3132 - loss: 1.5657 - val_accuracy: 0.3417 - val_loss: 1.4255
Epoch 2/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.4243 - loss: 1.3914 - val_accuracy: 0.4264 - val_loss: 1.3457
Epoch 3/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.4389 - loss: 1.4426 - val_accuracy: 0.4389 - val_loss: 1.2526
Epoch 4/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.4309 - loss: 1.8277 - val_accuracy: 0.4361 - val_loss: 1.8722
Epoch 5/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.3851 - loss: 3.4987 - val_accuracy: 0.3944 - val_loss: 2.7298
Epoch 6/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.3517 - loss: 6.6814 - val_accuracy: 0.3347 - val_loss: 5.1032
Epoch 7/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.3646 - loss: 8.2980 - val_accuracy: 0.5250 - val_loss: 2.1560
Epoch 8/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.4014 - loss: 8.0702 - val_accuracy: 0.

In [97]:
loss, acc = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("Augmented Delta CNN Accuracy:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4733 - loss: 2.0754
Augmented Delta CNN Accuracy: 0.47333332896232605
